# LLM Evaluation

Exact-match and F1 miss semantic equivalence. Human review does not scale. LLM-as-judge is the production answer -- with enough calibration to trust the number.

## Problem Definition

You need an evaluator that **understands meaning**, runs **cheaply** at scale, does not **lie** about regressions, and surfaces the right **failure modes**.

### Frameworks

1. RAGAS.   Retrieval-Augumented Generation ASsessment. Four RAG metrics (faithfulness, answer-relevance, context-precision, context-recall)

2. DeepEval.  Pytest for LLMs.

3. G-Eval.   A method: LLM-as-judge with chain-of-thought, custom criteria, 0-1 score

## Basic Concept

### LLM-as-judge

Replace a static metric with an LLM that scores outputs given a rubric. Given `(query, context, answer)` prompt a judge LLM: "Score 0-1 on faithfulness"

### Failure modes

1. Judge bias. Judegs prefer longer answers, answers from their own model family, answers that match the prompt style.

2. JSON parsing failures.  Bad JSON -> NAN score -> silently excluded from the aggreate.

3. Drfit over model version.  Freeze judge model + version plz.

# Build your Own

## Faithfulness

In [6]:
from typing import Callable
from transformers import pipeline
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

from langchain.chat_models import init_chat_model

nli = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-small",
    top_k=None,
)

def atomic_claims(answer: str, llm) -> list[str]:
    prompt = f"""Break this answer into simple factual claims (one per line):
            {answer}
        """
    return llm.invoke(prompt).content.splitlines()

def entailment_score(context: str, claim: str) -> float:
    result = nli({"text": context, "text_pair": claim})
    if isinstance(result, dict):
        scores = [result]
    elif isinstance(result, list) and result and isinstance(result[0], list):
        scores = result[0]
    else:
        scores = result

    entail = next((s for s in scores if s["label"] == "entailment"), None)
    return entail["score"] if entail else 0.0

def faithfulness(answer: str, context: str, llm) -> float:
    claims = [c.strip() for c in atomic_claims(answer, llm) if c.strip()]
    if not claims:
        return 0.0

    supported = 0
    for claim in claims:
        if entailment_score(context, claim) > 0.5:
            supported += 1
    return supported / len(claims)

with SectionPrinter("Faithfulness"):
    llm = init_chat_model("deepseek:deepseek-chat", extra_body={"thinking": {"type": "disabled"}})
    print(faithfulness("The capital of France is Paris", "France's capital is Paris", llm))
    
    
    
    
    

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

========================Faithfulness========================
1.0


## Answer relevance

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def answer_relevance(question: str, answer: str, encoder, llm, n = 3) -> float:
    prompt = f"Write {n} questions this answer could be the answer to:\n{answer}"
    generated = [line for line in llm.invoke(prompt).content.splitlines() if line.strip()][:n]
    print(generated)
    if not generated:
        return 0.0
    q_emb = np.asarray(encoder.encode([question], normalize_embeddings=True)[0])
    g_embs = np.asarray(encoder.encode(generated, normalize_embeddings=True))
    sims = [float(q_emb @ g_emb) for g_emb in g_embs]
    return sum(sims) / len(sims) 

with SectionPrinter("Answer relevance"):
    print(answer_relevance("What is the capital of France?", "Paris is the capital of France", encoder, llm))
    

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

======================Answer relevance======================
['Here are three questions this answer could be the answer to:', '1. **What is the capital city of France?**', '2. **In which city is the Eiffel Tower located?** (A slightly indirect way of asking, but it works if the question implies "What is the city that is also the capital of France?")']
0.45304786165555316


## G_Eval custom mertric

In [9]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter, load_project_env

from deepeval.metrics import GEval
from deepeval.models import DeepSeekModel
from deepeval.test_case import LLMTestCaseParams, LLMTestCase

load_project_env()

metric = GEval(
    name="Correctness",
    criteria="The answer should be factually accurate and match the expected output.",
    evaluation_steps=[
        "Read the expected output.",
        "Read the actual output.",
        "List factual claims in the actual output.",
        "For each claim, mark supported or unsupported by the expected output.",
        "Return score = fraction supported.",
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],
    model=DeepSeekModel(model="deepseek-chat"),
)

test = LLMTestCase(
    input="When was the first iPhone released?",
    actual_output="June 29th, 2007.",
    expected_output="June 29, 2007.",
)

with SectionPrinter("G-Eval custom metric"):
    metric.measure(test)
    print(metric.score, metric.reason)

/Users/keyficller/Documents/AIEngineering/.venv/lib/python3.14/site-packages/deepeval/evaluate/execute/loop.py:806: SyntaxWarning: 'return' in a 'finally' block
  return
/var/folders/6g/9chj8ddx1033kwmzkbfd95l40000gn/T/ipykernel_8581/775293789.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams, LLMTestCase


DeepEvalError: OpenAI API key is empty. Please configure a valid key.